# RAG Evaluation

**Module:** 04 — RAG

Measure retrieval and generation separately, then use LLM-as-judge carefully.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compute retrieval metrics like recall@k and MRR
- Define generation metrics: faithfulness, answer relevance, context precision
- Design LLM-as-judge prompts with rubrics and limitations
- Build a tiny offline eval loop


## Retrieval Metrics

**Definition.** Scores that judge whether the right evidence was retrieved, independent of wording.

**Why it matters.** If recall is poor, no prompt will reliably save you.

**How it works.** Label gold chunk/doc ids per question; compute recall@k, precision@k, MRR, nDCG.

**Intuition.** Did the right binder make the desk?

**Common pitfalls.**
- Gold labels only at doc level when chunking matters
- Tiny eval sets

**When to use.** Every iteration of chunking/index/query changes.

| Metric | Meaning |
|--------|---------|
| Recall@k | Fraction of gold items in top-k |
| MRR | 1/rank of first relevant |
| nDCG | Graded ranking quality |


In [ ]:
# Demo 1 — recall@k and MRR
def recall_at_k(gold, ranked, k):
    return len(set(gold) & set(ranked[:k])) / max(1, len(gold))

def mrr(gold, ranked):
    for i, doc in enumerate(ranked, 1):
        if doc in gold: return 1/i
    return 0.0

gold = {"C2", "C9"}
ranked = ["C5", "C2", "C1", "C9"]
print("R@3", recall_at_k(gold, ranked, 3), "MRR", mrr(gold, ranked))


In [ ]:
# Demo 2 — eval set structure
eval_set = [
    {"q": "refund window", "gold": ["pol-refund#3"]},
    {"q": "reset password", "gold": ["acct-security#1"]},
]
print(len(eval_set), eval_set[0])


In [ ]:
# Demo 3 — macro average
scores = [0.5, 1.0, 0.0]
print(sum(scores)/len(scores))


### Try it yourself — Retrieval Metrics

1. Label gold chunks for 10 questions in a toy corpus; compute R@5.


## Generation Metrics

**Definition.** Metrics on the answer: faithfulness to context, relevance to question, completeness.

**Why it matters.** Good retrieval can still yield ungrounded or incomplete answers.

**How it works.** Human rubrics, exact-match for short answers, semantic scores, groundedness checks.

**Intuition.** Right books, wrong essay—still fail.

**Common pitfalls.**
- BLEU on open answers
- Ignoring refusals that are correct

**When to use.** After retrieval metrics are non-terrible.


In [ ]:
# Demo 1 — crude faithfulness via entailment proxy (token support)
def faithfulness(answer, contexts):
    ctx = set(" ".join(contexts).lower().split())
    toks = [t for t in answer.lower().split() if t.isalpha() and len(t)>3]
    if not toks: return 1.0
    return sum(t in ctx for t in toks) / len(toks)
print(faithfulness("Refunds within sixty days", ["Refunds within sixty days of purchase"]))
print(faithfulness("Refunds within 365 days", ["Refunds within sixty days of purchase"]))


In [ ]:
# Demo 2 — answer relevance keyword proxy
def relevance(question, answer):
    q = set(question.lower().split()); a = set(answer.lower().split())
    return len(q & a) / max(1, len(q))
print(relevance("refund window", "The refund window is 60 days"))


In [ ]:
# Demo 3 — context precision proxy
def context_precision(useful_flags):
    # useful_flags[i] True if chunk i was needed
    hits = 0; prec = []
    for i, u in enumerate(useful_flags, 1):
        hits += int(u); prec.append(hits/i)
    return sum(p for p,u in zip(prec, useful_flags) if u) / max(1, sum(useful_flags))
print(context_precision([True, False, True]))


### Try it yourself — Generation Metrics

1. Create a 3-point rubric for faithfulness and score two answers by hand.


## LLM-as-Judge

**Definition.** Use an LLM with a rubric to score answers or retrieval usefulness.

**Why it matters.** Scales evaluation when exact match fails—but can be biased/noisy.

**How it works.** Fixed rubric JSON schema; blind positions; calibrate against humans; track judge model version.

**Intuition.** A grading TA—useful, not infallible.

**Common pitfalls.**
- Judge sharing task model biases
- Unstable prompts
- No human calibration

**When to use.** Large regression suites after human calibration on a seed set.

```mermaid
flowchart LR
  Q[Question] --> J[Judge LLM]
  C[Contexts] --> J
  A[Answer] --> J
  J --> S[Scores JSON]
  S --> H[Human calibrate]
```


In [ ]:
# Demo 1 — judge prompt + expected JSON
import json
YOUR_API_KEY = "YOUR_API_KEY"
judge_prompt = {
  "model": "gpt-4.1",
  "messages": [{
    "role": "user",
    "content": (
      "Score faithfulness 1-5. Return JSON keys: score, rationale.\n"
      "QUESTION: ...\nCONTEXT: ...\nANSWER: ..."
    ),
  }],
  "response_format": {"type": "json_object"},
}
print(json.dumps(judge_prompt, indent=2)[:500])
print("key", YOUR_API_KEY)


In [ ]:
# Demo 2 — parse judge output safely
def parse_judge(raw: str) -> dict:
    import json
    data = json.loads(raw)
    score = int(data.get("score", 0))
    if score < 1 or score > 5: raise ValueError("out of range")
    return {"score": score, "rationale": data.get("rationale", "")}
print(parse_judge('{"score": 4, "rationale": "mostly supported"}'))


In [ ]:
# Demo 3 — agreement with humans
human = [5, 4, 2, 4]
judge = [5, 3, 2, 5]
agree = sum(abs(h-j) <= 1 for h,j in zip(human, judge)) / len(human)
print("within-1 agreement", agree)


### Try it yourself — LLM-as-Judge

1. Write a faithfulness rubric with anchor examples for scores 1, 3, and 5.
2. Explain why the judge model should often differ from the task model.


## Glossary

- **recall@k**: Fraction of relevant items retrieved in top-k
- **faithfulness**: Answer supported by provided context


### Workshop drill — RAG Evaluation (1)

Diagram the data flow on paper, then implement one missing log line per stage.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 1 — RAG Evaluation
stages = ['ingest','chunk','embed','retrieve','pack','generate']
for s in stages:
    print(f'log.{{s}}.ok = ?')


### Workshop drill — RAG Evaluation (2)

Create two adversarial queries (one ID-heavy, one paraphrase-heavy) and compare hits.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 2 — RAG Evaluation
queries = ['error code E42-refund', 'how do I get my money back?']
for q in queries:
    print('Q:', q)
    print('  TODO: print top-3 ids')


### Workshop drill — RAG Evaluation (3)

Write a refusal test: empty hits must not produce a confident numeric answer.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 3 — RAG Evaluation
def must_refuse(hits):
    return (not hits) or hits[0].get('score',0) < 0.2
assert must_refuse([])
assert must_refuse([{'score': 0.05}])
assert not must_refuse([{'score': 0.9}])
print('refusal tests ok')


### Workshop drill — RAG Evaluation (4)

Estimate cost: vary top_k and context tokens; print a small table.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 4 — RAG Evaluation
rows = []
for k in [2,4,8,16]:
    toks = k*400
    rows.append((k, toks, round(toks/1e6*0.5, 5)))
print('k  ctx_toks  approx_$')
for r in rows:
    print(*r)


### Workshop drill — RAG Evaluation (5)

Add one metadata field and filter it in retrieval (tenant or product).

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 5 — RAG Evaluation
docs = [{'id':'a','tenant':'acme'},{'id':'b','tenant':'beta'}]
tenant='acme'
print([d for d in docs if d['tenant']==tenant])


## Summary & Key Takeaways

- Split retrieval metrics from generation metrics
- Gold labels make offline iteration possible
- LLM-as-judge needs rubrics and human calibration
- Ship regression eval in CI for RAG changes

### Practice

Create a 15-question eval JSON and score a toy retriever.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
